In [0]:
%pip install openpyxl

In [0]:
%sql
select distinct sheet_name
from sandbox.z_yeswook_kim.jira_1410_raw

-- %sql
-- create or replace table sandbox.z_yeswook_kim.jira_1410_l1
-- select 
--     sheet_name,
--     row_number,
--     CSN,
--     MAC_ADDRESS,
--     concat_ws(':',
--         substring(MAC_ADDRESS, 1, 2),
--         substring(MAC_ADDRESS, 3, 2),
--         substring(MAC_ADDRESS, 5, 2),
--         substring(MAC_ADDRESS, 7, 2),
--         substring(MAC_ADDRESS, 9, 2),
--         substring(MAC_ADDRESS, 11, 2)
--     ) as mac_addr_origin,
--     kic_data_ods.tlamp.hash_mac(mac_addr_origin) as mac_addr
-- from sandbox.z_yeswook_kim.jira_1410_raw
-- -- group by 
--     -- sheet_name

In [0]:
%sql
select sheet_name, count(1)
from sandbox.z_yeswook_kim.jira_1410_raw
group by sheet_name

In [0]:
%sql
SELECT
    sheet_name,
    count(1)                        AS total_rows,
    count(DISTINCT row_number)      AS unique_rows,
    count(1) - count(DISTINCT row_number) AS duplicate_count
FROM sandbox.z_yeswook_kim.jira_1410_raw
WHERE sheet_name IN ('MA1', 'MA2', 'MA3', 'MA4')
GROUP BY sheet_name
ORDER BY sheet_name

In [0]:
%sql
# 1. 중복 제거된 MA2/3/4 임시 테이블 저장
spark.sql("""
    CREATE OR REPLACE TABLE sandbox.z_yeswook_kim.jira_1410_raw_temp AS
    SELECT DISTINCT AFFILIATE_CODE, FROM_ORGANIZATION_CODE, PROD_DATE, PROD_MODEL,
           WORK_ORDER, PROD_LINE, CSN, MAC_ADDRESS, sheet_name, row_number
    FROM sandbox.z_yeswook_kim.jira_1410_raw
    WHERE sheet_name IN ('MA2', 'MA3', 'MA4')
""")
print("1/4 임시 테이블 생성 완료")

# 2. 원본에서 MA2/3/4 전체 삭제
spark.sql("""
    DELETE FROM sandbox.z_yeswook_kim.jira_1410_raw
    WHERE sheet_name IN ('MA2', 'MA3', 'MA4')
""")
print("2/4 MA2/3/4 삭제 완료")

# 3. 중복 제거된 데이터 재적재
spark.sql("""
    INSERT INTO sandbox.z_yeswook_kim.jira_1410_raw
    SELECT * FROM sandbox.z_yeswook_kim.jira_1410_raw_temp
""")
print("3/4 재적재 완료")

# 4. 임시 테이블 정리
spark.sql("DROP TABLE IF EXISTS sandbox.z_yeswook_kim.jira_1410_raw_temp")
print("4/4 임시 테이블 삭제 완료 — 중복 제거 완료")

In [0]:
%sql
SELECT sheet_name, count(1) AS cnt
FROM sandbox.z_yeswook_kim.jira_1410_raw
GROUP BY sheet_name
ORDER BY sheet_name

In [0]:
# import pandas as pd
# import openpyxl

# wb = openpyxl.load_workbook("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", read_only=True)
# sheet_names = wb.sheetnames
# display(sheet_names)

# for sheet_name in sheet_names:
#     df_sheet = pd.read_excel("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", sheet_name=sheet_name)
#     df_sheet['sheet_name'] = sheet_name
#     df_sheet['row_number'] = range(1, len(df_sheet) + 1)
#     spark_df = spark.createDataFrame(df_sheet)
#     spark_df.write.format("delta").mode("append").saveAsTable("sandbox.z_yeswook_kim.jira_1410_raw")

In [0]:
sheet_name

In [0]:
import pandas as pd
import openpyxl

wb = openpyxl.load_workbook("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", read_only=True)
sheet_names = wb.sheetnames
display(sheet_names)

for sheet_name in ('MA2','MA3','MA4'):
    df_sheet = pd.read_excel("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", sheet_name=sheet_name)
    df_sheet['sheet_name'] = sheet_name
    df_sheet['row_number'] = range(1, len(df_sheet) + 1)
    spark_df = spark.createDataFrame(df_sheet)
    spark_df.write.format("delta").mode("append").saveAsTable("sandbox.z_yeswook_kim.jira_1410_raw")

In [0]:
%sql
create or replace table sandbox.z_yeswook_kim.jira_1410_l1
select 
    sheet_name,
    row_number,
    CSN,
    MAC_ADDRESS,
    concat_ws(':',
        substring(MAC_ADDRESS, 1, 2),
        substring(MAC_ADDRESS, 3, 2),
        substring(MAC_ADDRESS, 5, 2),
        substring(MAC_ADDRESS, 7, 2),
        substring(MAC_ADDRESS, 9, 2),
        substring(MAC_ADDRESS, 11, 2)
    ) as mac_addr_origin,
    eic_data_ods.tlamp.hash_mac(mac_addr_origin) as mac_addr
from sandbox.z_yeswook_kim.jira_1410_raw
-- group by 
    -- sheet_name